In [ ]:
# Import modules
import numpy as np

# Import Sentence E,bedding Transformer
import textwrap as tw
from sentence_transformers import SentenceTransformer;

# Import RAG DB FAISS
import faiss

In [2]:
# Read input file & Create Sentence Embedding.
content = []
try:
    with open('C:\\RanjithC\\AIProjects\\PromptEngg\\huggingface\\data\\general_knowledge.txt', 'r') as file:
        content = file.readlines() # Reads the entire file content into a string
        for i, item in enumerate(content):
            content[i] = item.strip()
except Exception as e:
    print("An error occurred:", e)
print(content)

#Create Sentence Embedding 
model = SentenceTransformer('all-MiniLM-L6-v2',similarity_fn_name='cosine', device='cpu')

sentence_embedding = model.encode(content, batch_size=32, output_value='sentence_embedding', convert_to_numpy=True, device='cpu')
print(sentence_embedding.shape)
print(sentence_embedding[0])

['Capital of France is Paris.', 'The largest ocean on Earth is the Pacific Ocean.', 'Water boils at 100 degrees Celsius (212 degrees Fahrenheit) at sea level.', 'The human body has 206 bones.', 'The Great Wall of China is the longest man-made structure.', 'Mount Everest is the highest mountain in the world.', 'The chemical symbol for gold is Au.', 'A group of lions is called a pride.', 'The currency of Japan is the Japanese Yen.', 'The Earth revolves around the Sun.', 'The fastest land animal is the cheetah.', "The primary gas in Earth's atmosphere is nitrogen.", 'The inventor of the light bulb was Thomas Edison.', 'The longest river in the world is the Nile River.', 'The capital of Italy is Rome.', 'The process by which plants make their food is photosynthesis.', 'The human heart has four chambers.', 'The planet Mars is often called the "Red Planet".', 'The study of earthquakes is called seismology.', 'The square root of 144 is 12.', 'The largest desert in the world is the Sahara Dese

In [3]:
# Add Sentence Embedding to FAISS VectorDB
#Set dimension of VectorDB Index and return the Index
#Dimension of Vectors (number of cols) can be got from shape[1]
#Here the VectorDB Index size is 384 dimension
fs_vector_index = faiss.IndexFlatL2(sentence_embedding.shape[1])

#Add embeddings to the above index
fs_vector_index.add(sentence_embedding)

print("number of vectors: ", fs_vector_index.ntotal)


number of vectors:  2135


In [ ]:
# Do Search on above FAISS VectorDB

def do_search(query):
    query_embedding = model.encode(query, batch_size=32, output_value='sentence_embedding', convert_to_numpy=True, device='cpu')
    print(query_embedding.shape)

    #Convert query_embedding to a 2D Vector of 1 Row and 1 column of 384 dim array
    query_embedding = query_embedding[np.newaxis, :]
    print(query_embedding.shape)

    # Return tuple (Euc Dist, Vector Index / Position) of 'k' nearest neighbours to the query embedding
    # top_dist and top_pos is an Array with same number of rows. 
    top_dist, top_pos = fs_vector_index.search(query_embedding, k=25)

    print('Shortest Euc dist: ', top_dist)
    print('Position of dist: ', top_pos)

    return(top_pos) 

In [12]:
# input questions

query = ''

query = input("Enter your question: ")
print(query)

result_arr = do_search(query)
for i in result_arr[0]:
    print(content[i])


Get list of chemical elements from teh dataset
(384,)
(1, 384)
Shortest Euc dist:  [[1.3037866 1.3059651 1.3214122 1.3251321 1.3368673 1.3423584 1.3431298
  1.3502817 1.3529487 1.3555443 1.3577305 1.3580081 1.358604  1.358604
  1.3660846 1.3701822 1.3712091 1.3728571 1.3728576 1.3766648 1.3772926
  1.3787928 1.380252  1.3900189 1.3907368]]
Position of dist:  [[2057  497  761  537 1353  993 1297  385 2041 2009 1313  569   93  361
   281  929 1817  457  417 1873  793 2001  425 1521 2121]]
The chemical symbol for bioctseptium is Bos.
The chemical symbol for technetium is Tc.
The chemical symbol for osmium is Os.
The chemical symbol for cadmium is Cd.
The chemical symbol for unpentoctium is Upo.
The chemical symbol for dubnium is Db.
The chemical symbol for unpentunium is Upu.
The chemical symbol for copper is Cu.
The chemical symbol for bioctpentium is Bop.
The chemical symbol for bioctunium is Bou.
The chemical symbol for unpenttrium is Upt.
The chemical symbol for tellurium is Te.
The c